# Feature Engineering - Hotel Cancellation / Oversell Project

This notebook holds all the data preparation that used to sit inside `Model.ipynb`.
After running it, the data is split into Train / Choose / Proof periods and the
preprocessing pipelines are ready to plug into the models.

**Steps**
1. Load the data and fix data types
2. Split by time (Train / Choose / Proof)
3. Choose the features the models may use
4. Build the preprocessing pipelines
5. Create X and y for each period
6. Fit the preprocessors on Train and check the output

**Input file:** `hotels_model_ready.csv` (keep it in the same folder as this notebook)

In [1]:
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, StandardScaler

In [2]:
df = pd.read_csv("hotels_model_ready.csv", keep_default_na=False)

# Make sure every column has the right type
df["arrival_date"] = pd.to_datetime(df["arrival_date"])
df["children"] = df["children"].astype(float).astype(int)
df["total_guests"] = df["total_guests"].astype(float).astype(int)

# agent and company are ID numbers, but they are categories, not quantities
df["agent"] = df["agent"].astype(str)
df["company"] = df["company"].astype(str)

print(f"Loaded shape : {df.shape[0]:,} rows x {df.shape[1]} columns")
print(f"Date range   : {df['arrival_date'].min().date()} to {df['arrival_date'].max().date()}")
print(f"Missing cells: {df.isnull().sum().sum()}")
df.head(3)

Loaded shape : 119,390 rows x 28 columns
Date range   : 2015-07-01 to 2017-08-31
Missing cells: 0


,hotel,is_canceled,lead_time,arrival_date_year,arrival_date_month,arrival_date_week_number,arrival_date_day_of_month,stays_in_weekend_nights,stays_in_week_nights,adults,...,previous_bookings_not_canceled,reserved_room_type,deposit_type,agent,company,customer_type,adr,nights,total_guests,arrival_date
0,Resort Hotel,0,342,2015,July,27,1,0,0,2,...,0,C,No Deposit,None,None,Transient,0.0,0,2,2015-07-01
1,Resort Hotel,0,737,2015,July,27,1,0,0,2,...,0,C,No Deposit,None,None,Transient,0.0,0,2,2015-07-01
2,Resort Hotel,0,7,2015,July,27,1,0,1,1,...,0,A,No Deposit,None,None,Transient,75.0,1,1,2015-07-01


In [3]:
TRAIN_END = "2016-07-01"
PROOF_START = "2017-01-01"

def split_periods(df):
    train = df[df["arrival_date"] < TRAIN_END].copy()
    choose = df[(df["arrival_date"] >= TRAIN_END) & (df["arrival_date"] < PROOF_START)].copy()
    proof = df[df["arrival_date"] >= PROOF_START].copy()
    return train, choose, proof

train, choose, proof = split_periods(df)

print(f"Train  (< 2016-07-01) : {len(train):,} bookings")
print(f"Choose (2016 H2)      : {len(choose):,} bookings")
print(f"Proof  (2017 Jan-Aug) : {len(proof):,} bookings")

Train  (< 2016-07-01) : 49,157 bookings
Choose (2016 H2)      : 29,546 bookings
Proof  (2017 Jan-Aug) : 40,687 bookings


In [4]:
TARGET = "is_canceled"

NUMERIC_FEATURES = [
    "lead_time", "arrival_date_year", "arrival_date_day_of_month", "arrival_date_week_number",
    "stays_in_weekend_nights", "stays_in_week_nights", "nights", "adults", "children",
    "babies", "total_guests", "is_repeated_guest", "previous_cancellations",
    "previous_bookings_not_canceled", "adr",
]

CATEGORICAL_FEATURES = [
    "hotel", "arrival_date_month", "meal", "country", "market_segment",
    "distribution_channel", "reserved_room_type", "deposit_type",
    "agent", "company", "customer_type",
]

FEATURE_COLS = NUMERIC_FEATURES + CATEGORICAL_FEATURES

# Safety checks: no target or contract date inside the features
assert TARGET not in FEATURE_COLS
assert "arrival_date" not in FEATURE_COLS

print(f"Total features: {len(FEATURE_COLS)} "
      f"({len(NUMERIC_FEATURES)} numeric, {len(CATEGORICAL_FEATURES)} categorical)")

Total features: 26 (15 numeric, 11 categorical)


In [5]:
prep_linear = ColumnTransformer([
    ("num", Pipeline([
        ("imp", SimpleImputer(strategy="median")),
        ("sc", StandardScaler()),
    ]), NUMERIC_FEATURES),
    ("cat", OneHotEncoder(handle_unknown="ignore", min_frequency=20), CATEGORICAL_FEATURES),
])

prep_tree = ColumnTransformer([
    ("num", SimpleImputer(strategy="median"), NUMERIC_FEATURES),
    ("cat", OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1),
     CATEGORICAL_FEATURES),
])

## 5. Create X (features) and y (target) for each period

In [6]:
X_train, y_train = train[FEATURE_COLS], train[TARGET]
X_choose, y_choose = choose[FEATURE_COLS], choose[TARGET]
X_proof, y_proof = proof[FEATURE_COLS], proof[TARGET]

print(f"X_train  {X_train.shape}   X_choose {X_choose.shape}   X_proof {X_proof.shape}")
print(f"\nCancellation rate -> train {y_train.mean():.1%}, "
      f"choose {y_choose.mean():.1%}, proof {y_proof.mean():.1%}")

X_train  (49157, 26)   X_choose (29546, 26)   X_proof (40687, 26)

Cancellation rate -> train 35.8%, choose 36.9%, proof 38.7%


In [7]:
# Linear version
Xl_train = prep_linear.fit_transform(X_train)
Xl_choose = prep_linear.transform(X_choose)
Xl_proof = prep_linear.transform(X_proof)
print("Linear preprocessor ->", Xl_train.shape, Xl_choose.shape, Xl_proof.shape)

# Tree version
Xt_train = prep_tree.fit_transform(X_train)
Xt_choose = prep_tree.transform(X_choose)
Xt_proof = prep_tree.transform(X_proof)
print("Tree preprocessor   ->", Xt_train.shape, Xt_choose.shape, Xt_proof.shape)

print("\nFeature engineering OK.")

Linear preprocessor -> (49157, 275) (29546, 275) (40687, 275)
Tree preprocessor   -> (49157, 26) (29546, 26) (40687, 26)

Feature engineering OK.
